<a href="https://colab.research.google.com/github/twillixa/Algorithm-in-BI/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
!pip install odfpy -q
!git clone https://github.com/twillixa/Algorithm-in-BI.git
%cd Algorithm-in-BI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 22.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Cloning into 'Algorithm-in-BI'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 16 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 169.86 KiB | 18.87 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Algorithm-in-BI/Algorithm-in-BI


##Loading and Cleaning

In [7]:
clients  = pd.read_excel("RawData/clients.ods",  engine="odf")
comandes = pd.read_excel("RawData/comandes.ods", engine="odf")

clients.to_csv("RawData/clients.csv",   index=False, encoding="utf-8-sig")
comandes.to_csv("RawData/comandes.csv", index=False, encoding="utf-8-sig")

print(clients.head())
print(f"comandes → {comandes.shape}")

  PRENOM  NOM  EMAIL CODE POSTAL ANNÉE DE NAISSANCE
0   Abel  NaN    NaN        1003         30.08.1994
1   Adam  NaN    NaN        1022         23.01.1999
2   Adam  NaN    NaN        2740               2003
3  Adela  NaN    NaN        1007         03.04.1976
4  Adèle  NaN    NaN        1040               2003
comandes → (2039, 11)


In [8]:
# ============================================================
# CLEANING
# ============================================================

#  Drop unnecessary columns
comandes.drop(columns=["NOM", "EMAIL", "DEVISE"], inplace=True)

#  Parse DATE COMMANDE as datetime
comandes["DATE COMMANDE"] = pd.to_datetime(comandes["DATE COMMANDE"])

#  Standardize ANNEE DE NAISSANCE → extract year as integer
comandes["ANNEE DE NAISSANCE"] = (
    comandes["ANNEE DE NAISSANCE"]
    .astype(str)
    .str.extract(r"(\d{4})")
    .astype("Int64")  # nullable integer to handle NaNs
)

#  Cast CODE POSTAL to string
comandes["CODE POSTAL"] = (
    comandes["CODE POSTAL"]
    .astype(str)
    .str.strip()
    .replace("nan", pd.NA)
)

#  Flag duplicate CODE COMMANDE
comandes["IS_DUPLICATE"] = comandes.duplicated(subset="CODE COMMANDE", keep=False)
print(f"Duplicate orders flagged: {comandes['IS_DUPLICATE'].sum()}")

#  Categorical variable for MODE DE PAIEMENT
comandes["MODE DE PAIEMENT"] = pd.Categorical(comandes["MODE DE PAIEMENT"])
print("Payment categories:", comandes["MODE DE PAIEMENT"].cat.categories.tolist())

#  Compute age
comandes["AGE"] = 2026 - comandes["ANNEE DE NAISSANCE"]
print(comandes["AGE"].describe())


# Preview
print(comandes.dtypes)
display(comandes.head())

Duplicate orders flagged: 0
Payment categories: ['Caisse - Carte de crédit', 'Carte de crédit', 'Gratuit', 'Postcard', 'Twint']
count       1975.0
mean     43.588861
std      95.003253
min        -3055.0
25%           29.0
50%           41.0
75%           55.0
max         1023.0
Name: AGE, dtype: Float64
PRENOM                        object
CODE COMMANDE                  int64
DATE COMMANDE         datetime64[ns]
MONTANT                        int64
MODE DE PAIEMENT            category
NB BILLETS                     int64
CODE POSTAL                   object
ANNEE DE NAISSANCE             Int64
IS_DUPLICATE                    bool
AGE                            Int64
dtype: object


,PRENOM,CODE COMMANDE,DATE COMMANDE,MONTANT,MODE DE PAIEMENT,NB BILLETS,CODE POSTAL,ANNEE DE NAISSANCE,IS_DUPLICATE,AGE
0,fiona,69863303,2025-12-16 13:55:55,142,Carte de crédit,3,<NA>,1972,False,54
1,Natascha,69865565,2025-12-16 16:19:31,93,Twint,2,1023,1964,False,62
2,Martine,69866924,2025-12-16 17:28:10,44,Twint,1,1510,1964,False,62
3,Pierre-Philippe,69866513,2025-12-16 17:32:24,88,Carte de crédit,2,1033,1955,False,71
4,Sandra,69869702,2025-12-16 19:57:18,49,Twint,1,1022,1976,False,50
